# Part 2: Basic SQL Queries

In this notebook we practice the most common SQL query patterns:

- `SELECT` / `WHERE` / `ORDER BY` / `LIMIT`
- `LIKE`, `IN`, `BETWEEN`
- `GROUP BY` / `COUNT` / `AVG`

> **Prerequisite:** Run `01_setup.ipynb` first to create `gene_expression.db`.

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("gene_expression.db")
print("Connected.")

Connected.


## Helper function

`query()` runs SQL and returns a DataFrame for easy viewing.

In [2]:
def query(sql, params=()):
    return pd.read_sql(sql, con, params=params)

---
## 1. SELECT — retrieve all genes

In [3]:
query("""
SELECT symbol, chromosome, strand, start_pos, end_pos
FROM   genes
ORDER BY chromosome, start_pos
""")

,symbol,chromosome,strand,start_pos,end_pos
0,PTEN,10,-,89622870,89731687
1,ATM,11,+,108222832,108369102
2,KRAS,12,-,25205246,25250929
3,BRCA2,13,+,32315474,32400266
4,RB1,13,-,48303747,48481890
5,AKT1,14,+,104769349,104795751
6,CDH1,16,-,68737275,68835822
7,TP53,17,-,7661779,7687550
8,HER2,17,+,39687914,39730426
9,BRCA1,17,+,43044295,43125483


---
## 2. WHERE — filter by chromosome

In [4]:
query("""
SELECT symbol, chromosome, strand
FROM   genes
WHERE  chromosome = '17'
""")

,symbol,chromosome,strand
0,BRCA1,17,+
1,TP53,17,-
2,HER2,17,+


---
## 3. ORDER BY and LIMIT — top 5 by genomic start position

In [5]:
query("""
SELECT symbol, chromosome, start_pos
FROM   genes
ORDER BY start_pos
LIMIT 5
""")

,symbol,chromosome,start_pos
0,TP53,17,7661779
1,VHL,3,10141777
2,KRAS,12,25205246
3,BRCA2,13,32315474
4,MLH1,3,36993459


---
## 4. LIKE — pattern matching on gene symbol

In [6]:
# % matches any sequence of characters
query("""
SELECT symbol, description
FROM   genes
WHERE  symbol LIKE 'BR%'
""")

,symbol,description
0,BRCA1,Breast cancer type 1 susceptibility protein
1,BRCA2,Breast cancer type 2 susceptibility protein


---
## 5. IN — match against a list of values

In [7]:
query("""
SELECT symbol, chromosome
FROM   genes
WHERE  symbol IN ('TP53', 'KRAS', 'MYC', 'PTEN')
""")

,symbol,chromosome
0,KRAS,12
1,MYC,8
2,PTEN,10
3,TP53,17


---
## 6. BETWEEN — range filter on expression values

In [8]:
query("""
SELECT g.symbol, s.name, e.raw_counts, e.tpm
FROM   expression e
JOIN   genes      g ON g.gene_id   = e.gene_id
JOIN   samples    s ON s.sample_id = e.sample_id
WHERE  e.raw_counts BETWEEN 500 AND 700
ORDER BY e.raw_counts
LIMIT 10
""")

,symbol,name,raw_counts,tpm
0,PIK3CA,LUNG_T02,500,307.93
1,EGFR,SKIN_T01,511,422.50
2,VHL,BRCA_N01,519,458.37
3,BRCA2,LUNG_T01,579,469.55
4,EGFR,BRCA_T02,579,814.55
5,ATM,BRCA_T01,594,799.53
6,AKT1,SKIN_T01,648,841.09
7,MYC,LUNG_T01,656,854.33
8,MYC,BRCA_T03,674,1473.16


---
## 7. GROUP BY + COUNT — genes per chromosome

In [9]:
query("""
SELECT chromosome, COUNT(*) AS gene_count
FROM   genes
GROUP BY chromosome
ORDER BY gene_count DESC
""")

,chromosome,gene_count
0,3,3
1,17,3
2,13,2
3,8,1
4,7,1
5,16,1
6,14,1
7,12,1
8,11,1
9,10,1


---
## 8. GROUP BY + AVG — mean expression by sample condition

In [10]:
query("""
SELECT s.condition,
       ROUND(AVG(e.tpm), 2) AS avg_tpm,
       COUNT(*)             AS n_measurements
FROM   expression e
JOIN   samples    s ON s.sample_id = e.sample_id
GROUP BY s.condition
""")

,condition,avg_tpm,n_measurements
0,normal,1505.39,60
1,tumor,1533.38,90


---
## Exercises

Write SQL queries to answer the following. Add new cells below each question.

**Q1.** How many genes are on chromosome 3?

In [ ]:
# Your query here


**Q2.** List all genes on the minus (`-`) strand, ordered alphabetically by symbol.

In [ ]:
# Your query here


**Q3.** How many expression measurements have a `tpm` greater than 1000?

In [ ]:
# Your query here


**Q4.** Which tissue type has the highest average TPM?

In [ ]:
# Your query here


In [ ]:
con.close()